## Cleaning the Data

**Step 1: Decide which columns may provide valuable insight to my business question**

- Keep (potential valuable insight):
    - id ... unique identifier
    - host_id ... identify host
    - neighbourhood ... location factor
    - latitude / longitude ... useful for Power BI map
    - room_type ... major price driver
    - price ... main variable being analyzed
    - minimum_nights ... pricing factor
    - number_of_reviews ... popularity proxy
    - reviews_per_month ... activity level
    - calculated_host_listings_count ... professional vs casual host
    - availability_365 ... availability factor

- Drop (no valuable insight at all):
    - name ... listing title, not useful for analysis
    - host_name ... not relevant to pricing
    - neighbourhood_group ... entirely empty across all three cities
    - last_review ... not relevant to business question
    - number_of_reviews_ltm ... redundant with number_of_reviews
    - license ... entirely empty across all three cities
    - host_profile_id ... Dallas only, not needed

In [20]:
import pandas as pd

# Load all three datasets
austin = pd.read_csv('../data/listings_Austin.csv')
dallas = pd.read_csv('../data/listings_Dallas.csv')
fort_worth = pd.read_csv('../data/listings_FortWorth.csv')

# store all there Columns to drop in an array to avoid repetition
cols_to_drop = [
    'name',
    'host_name',
    'neighbourhood_group',
    'last_review',
    'number_of_reviews_ltm',
    'license'
]

# Drop the columns from each dataset
austin = austin.drop(columns=cols_to_drop)

# Dallas has an additional column 'host_profile_id' that needs to be dropped
dallas = dallas.drop(columns=['host_profile_id'] + cols_to_drop)
fort_worth = fort_worth.drop(columns=cols_to_drop)

# Check to see if the columns are dropped
print(austin.shape)
print(dallas.shape)
print(fort_worth.shape)

(10533, 12)
(6217, 12)
(2081, 12)


**Step 2: Create a new column to determine which city each listing belongs to**
- When combining all 3 datasets together we need a way to distinguish whether the listing data is from Austin, Dallas, or Fort Worth

In [21]:
# Add a new column 'city' to each dataset to identify the city
austin['city'] = 'Austin'
dallas['city'] = 'Dallas'
fort_worth['city'] = 'Fort Worth'

# Check to see if the new column is added
print(austin.shape)
print(dallas.shape)
print(fort_worth.shape)

(10533, 13)
(6217, 13)
(2081, 13)


**Step 3: Handle Null Values**
- Depending on the column and its importance to our analysis, we need to determine the best approach for handling NULL values
- If the number of NULL values is small relative to the size of the dataset, dropping those rows entirely is more effective than trying to normalize the data
- If a column has many NaN values across the entire dataset, filling with a standardized value is more effective in order to retain the other data that the row may contain
- Special case: Fort Worth's neighbourhood column had 956 NULL values out of 2,081 rows — dropping these would eliminate ~46% of the dataset, so missing values were filled with 'Unknown' and flagged as a data limitation

In [22]:
# Drop rows where price is null
austin = austin.dropna(subset=['price'])
dallas = dallas.dropna(subset=['price'])
fort_worth = fort_worth.dropna(subset=['price'])

# Fill reviews_per_month nulls with 0
austin['reviews_per_month'] = austin['reviews_per_month'].fillna(0)
dallas['reviews_per_month'] = dallas['reviews_per_month'].fillna(0)
fort_worth['reviews_per_month'] = fort_worth['reviews_per_month'].fillna(0)

# Verify no more nulls
print(austin.isnull().sum())
print(dallas.isnull().sum())
print(fort_worth.isnull().sum())

id                                0
host_id                           0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
city                              0
dtype: int64
id                                0
host_id                           6
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    1
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    6
availability_365                  0
city                              0
dtype: int64
id                                  0


In [23]:
# Dallas has a small amount of remaining nulls in 
# 'host_id', 'minimum_nights', and 'calculated_host_listings_count' columns
# These rows need to be dropped
dallas = dallas.dropna(subset=['host_id', 'minimum_nights', 'calculated_host_listings_count'])

# Fill Fort Worth missing neighbourhoods with 'Unknown'
fort_worth['neighbourhood'] = fort_worth['neighbourhood'].fillna('Unknown')

# Re-Verify all nulls are handled
print('Austin nulls:')
print(austin.isnull().sum())
print('\nDallas nulls:')
print(dallas.isnull().sum())
print('\nFort Worth nulls:')
print(fort_worth.isnull().sum())

Austin nulls:
id                                0
host_id                           0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
city                              0
dtype: int64

Dallas nulls:
id                                0
host_id                           0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
city                              0
dtype: int64

Fort Wor

**Step 4: Removing Outliers from Airbnb Prices**
- From our earlier EDA we identified that some listings had prices up to $50,000 per night, which is clearly noise in the data
- Although some high-end listings may be legitimate, we want to focus on the realistic market range that represents the vast majority of listings
- Using the 99th percentile as a reference point, we found that 99% of listings across all three cities fall below $2,500 per night
- A uniform price cap of $1,500 was applied across all three cities to remove extreme outliers while still capturing the high-end market
- A consistent cap was chosen to ensure fair and unbiased city to city price comparisons

In [24]:
# Check the 99th percentile of price for each city to identify outliers
print(austin['price'].quantile(0.99))
print(dallas['price'].quantile(0.99))
print(fort_worth['price'].quantile(0.99))

2431.9200000000055
1886.9499999999753
1188.4000000000005


In [25]:
# Cap price at $1,500 across all three cities
austin = austin[austin['price'] <= 1500]
dallas = dallas[dallas['price'] <= 1500]
fort_worth = fort_worth[fort_worth['price'] <= 1500]

# Verify
print(austin.shape)
print(dallas.shape)
print(fort_worth.shape)

(10355, 13)
(5809, 13)
(1839, 13)


**Step 5: Removing Outliers from Airbnb Minimum Night Requirements**
- From our earlier EDA we identified that some listings had a minimum night requirement of up to 365 nights, which represents a full year stay and is clearly not a true short term rental
- To understand the realistic distribution we analyzed the most common minimum night values and found two distinct markets:
    - Short term rentals: 1 to 14 nights ... traditional Airbnb stays
    - Monthly rentals: 28 to 31 nights ... longer term stays
- A uniform cap of 30 nights was applied across all three cities to remove extreme outliers while still capturing both the short term and monthly rental markets
- A consistent cap was chosen to ensure fair and unbiased city to city comparisons
- Note: Fort Worth had 470 listings above 14 nights ... capping at 30 instead of 14 was partially motivated by protecting Fort Worth's already small dataset

In [26]:
# Cap minimum_nights at 30 across all three cities
austin = austin[austin['minimum_nights'] <= 30]
dallas = dallas[dallas['minimum_nights'] <= 30]
fort_worth = fort_worth[fort_worth['minimum_nights'] <= 30]

# Verify
print(austin.shape)
print(dallas.shape)
print(fort_worth.shape)

(10020, 13)
(5697, 13)
(1817, 13)


**Step 6: Converting Columns to Appropriate Data Types**
- Re-examining data types identified during EDA to ensure each column is stored as the correct type
- Austin: neighbourhood column converted from int64 to string since zip codes are categorical identifiers, not numbers
- Dallas: host_id, minimum_nights, and calculated_host_listings_count converted from float64 to int64 since these columns represent whole numbers and should never contain decimals

In [27]:
# Austin
austin['neighbourhood'] = austin['neighbourhood'].astype(str)

# Dallas
dallas['host_id'] = dallas['host_id'].astype(int)
dallas['minimum_nights'] = dallas['minimum_nights'].astype(int)
dallas['calculated_host_listings_count'] = dallas['calculated_host_listings_count'].astype(int)

# Verify
print("Austin:")
print(austin.dtypes)
print("\nDallas:")
print(dallas.dtypes)

Austin:
id                                  int64
host_id                             int64
neighbourhood                         str
latitude                          float64
longitude                         float64
room_type                             str
price                             float64
minimum_nights                      int64
number_of_reviews                   int64
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
city                                  str
dtype: object

Dallas:
id                                  int64
host_id                             int64
neighbourhood                         str
latitude                          float64
longitude                         float64
room_type                             str
price                             float64
minimum_nights                      int64
number_of_reviews                   int64
reviews_per_month                 float64
cal

**Step 7: Combining All 3 Datasets Into One**
- Now that each city's dataset has been individually cleaned, we can safely combine them into one unified dataset
- We use pd.concat() to stack all three dataframes on top of each other
- The city column we added in Step 2 ensures every row retains its origin city after combining, allowing us to distinguish and compare cities during SQL analysis
- ignore_index=True resets the row index to run continuously from 0 instead of restarting at 0 for each city

In [28]:
# Combine all three cities
df_combined = pd.concat([austin, dallas, fort_worth], ignore_index=True)

# Verify
print(df_combined.shape)
print(df_combined['city'].value_counts())

(17534, 13)
city
Austin        10020
Dallas         5697
Fort Worth     1817
Name: count, dtype: int64


**Step 8: Normalizing Column Names and String Values**
- Before exporting, we normalize all column names to lowercase and strip any leading or trailing whitespace to ensure consistency when loading into PostgreSQL
- String columns such as room_type, neighbourhood, and city are also stripped of whitespace and converted to title case to ensure consistent grouping and filtering during SQL analysis

In [29]:
# Normalize column names
df_combined.columns = df_combined.columns.str.lower().str.strip()

# Normalize string values
df_combined['room_type'] = df_combined['room_type'].str.strip().str.title()
df_combined['neighbourhood'] = df_combined['neighbourhood'].str.strip().str.title()
df_combined['city'] = df_combined['city'].str.strip().str.title()

# Verify
print(df_combined.shape)
print(df_combined.isnull().sum())
print(df_combined['city'].value_counts())

(17534, 13)
id                                0
host_id                           0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
city                              0
dtype: int64
city
Austin        10020
Dallas         5697
Fort Worth     1817
Name: count, dtype: int64


**Step 9: Export the Cleaned Dataset to CSV**
- Once all cleaning steps have been applied to the combined dataset, we export it to a new CSV file called cleaned_texas_listings.csv
- This file will be loaded into PostgreSQL for SQL analysis
- index=False is used to prevent pandas from writing the row numbers as an extra column in the CSV file

In [30]:
df_combined.to_csv('../data/cleaned_texas_listings.csv', index=False)

**Step 10: Verify the Cleaned Dataset**
- After exporting, we reload the cleaned CSV to confirm the file was saved correctly and all cleaning steps were applied successfully
- We verify the following:
    - Shape ... correct number of rows and columns
    - Null values ... zero nulls across all columns
    - City distribution ... correct number of listings per city
    - Data types ... all columns stored as the correct type

In [31]:
# Load the cleaned CSV and verify
df_check = pd.read_csv('../data/cleaned_texas_listings.csv')

print(df_check.shape)
print(df_check.info())
print(df_check['city'].value_counts())
print(df_check.isnull().sum())

(17534, 13)
<class 'pandas.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              17534 non-null  int64  
 1   host_id                         17534 non-null  int64  
 2   neighbourhood                   17534 non-null  str    
 3   latitude                        17534 non-null  float64
 4   longitude                       17534 non-null  float64
 5   room_type                       17534 non-null  str    
 6   price                           17534 non-null  float64
 7   minimum_nights                  17534 non-null  int64  
 8   number_of_reviews               17534 non-null  int64  
 9   reviews_per_month               17534 non-null  float64
 10  calculated_host_listings_count  17534 non-null  int64  
 11  availability_365                17534 non-null  int64  
 12  city                           